# Apprenticeship Data Cleaning

**Source:** `national_apprenticeships_raw.csv` — RAPIDS federal dataset  
**Output:** `national_apprenticeships_clean.csv`, `hawaii_apprenticeships_clean.csv`  

This notebook cleans and prepares the national apprenticeship dataset for analysis.
It does not merge or analyze — that happens in `02_data_analysis.ipynb`.

**Sections:**

0. Remove duplicate rows  
1. Drop administrative ID columns  
2. Program characteristic columns  
3. Apprentice status columns  
4. Demographic columns  
5. Occupation & industry columns  
6. Geography columns  
7. Date & time columns
8. Export

In [3]:
import numpy as np
import pandas as pd
import plotly.express as px

In [4]:
nation_data = pd.read_csv('../data/raw/national_apprenticeships_raw.csv', engine='pyarrow')
print(f"Rows: {len(nation_data):,} | Columns: {nation_data.shape[1]}")

Rows: 1,495,085 | Columns: 71


## 0. Remove Duplicate Rows

In [5]:
before = len(nation_data)
nation_data = nation_data.drop_duplicates(keep='first')
removed = before - len(nation_data)

print(f"Removed: {removed:,} duplicate rows")
print(f"Before:  {before:,} | After: {len(nation_data):,}")

Removed: 38,802 duplicate rows
Before:  1,495,085 | After: 1,456,283


## 1. Administrative ID Columns

These are system-generated keys used for linking records — not useful for analysis.

In [4]:
identity_cols = [
    'APPRENTICE_ID', 'APPRENTICE_NUMBER',
    'PROGRAM_NUMBER', 'EMPLOYER_ID', 'SAA_PORTAL_UNIQUE_ID'
]
nation_data = nation_data.drop(columns=identity_cols)
print(f"Dropped: {identity_cols}")

Dropped: ['APPRENTICE_ID', 'APPRENTICE_NUMBER', 'PROGRAM_NUMBER', 'EMPLOYER_ID', 'SAA_PORTAL_UNIQUE_ID']


## 2. Program Characteristic Columns

These describe the program itself: name, type, registration level, and location.

- `PROGRAM_ID` is the reliable unique identifier
- `PROGRAM_NAME` is human-typed with inconsistencies — use for display only, not filtering

### Data quality check

In [5]:
null_names = nation_data.groupby('PROGRAM_ID')['PROGRAM_NAME'].nunique()

print(f"Total unique program IDs:   {nation_data['PROGRAM_ID'].nunique():,}")
print(f"Total unique program names: {nation_data['PROGRAM_NAME'].nunique():,}")
print(f"IDs missing program name:   {(null_names == 0).sum():,}")
print(f"Names with multiple IDs:    {(null_names > 1).sum():,}")

Total unique program IDs:   26,974
Total unique program names: 25,942
IDs missing program name:   13
Names with multiple IDs:    0


The gap between IDs and names has two reaons:
- Large organizations register multiple programs under one name (e.g. PG&E has 33 IDs)
- 13 IDs are missing, all from Idaho or Indianapolis

In [6]:
# Confirm nameless IDs are CA/CO only — all other fields intact
# Only the program name itself failed to sync with RAPIDS
nameless_ids = null_names[null_names == 0].index.tolist()
nation_data[nation_data['PROGRAM_ID'].isin(nameless_ids)][['STATE', 'PROGRAM_NAME']].head()

,STATE,PROGRAM_NAME
23335,ID,None
59767,None,None
130282,None,None
281611,ID,None
323668,IN,None


### Cleaning

In [7]:
# Convert PROGRAM_ZIPCODE to category
# Int64 first to handle NaN, then string to strip whitespace, then category
nation_data['PROGRAM_ZIPCODE'] = (
    nation_data['PROGRAM_ZIPCODE']
    .astype('Int64')
    .astype('string')
    .str.strip()
    .astype('category')
)

assert nation_data['PROGRAM_ZIPCODE'].dtype == 'category', \
    "PROGRAM_ZIPCODE should be category dtype"

In [8]:
# Sanity check
program_cols = [
    'PROGRAM_ID', 'PROGRAM_NAME',
    'NATIONAL_PROGRAM', 'USMAP_PROGRAM', 'FBOP_PROGRAM',
    'PROGRAM_VIEW', 'PROGRAM_ZIPCODE'
]
nation_data[program_cols].sample(3)

,PROGRAM_ID,PROGRAM_NAME,NATIONAL_PROGRAM,USMAP_PROGRAM,FBOP_PROGRAM,PROGRAM_VIEW,PROGRAM_ZIPCODE
498285,102908,CITY BARBERING AND COSMETOLOGY APPRENTICESHIP ...,0,0,0,State,95828
950071,103046,LA MODA APPRENTICESHIP ACADEMY,0,0,0,State,95340
185251,3081,MIAMI ELECTRICAL JOINT APPRENTICESHIP AND TRAI...,0,0,0,State,33125


## 3. Apprentice Status Columns

Status codes are abbreviated — replacing with readable labels.

In [9]:
# Standardize and map status codes to readable labels
nation_data['APPRENTICE_STATUS_CODE'] = (
    nation_data['APPRENTICE_STATUS_CODE']
    .astype('string')
    .str.strip()
    .str.upper()
)

status_dict = {
    'CO': 'Completed',
    'CA': 'Cancelled',
    'RE': 'Active',
    'SU': 'Suspended',
    'RI': 'Reinstated'
}
nation_data['APPRENTICE_STATUS_CODE'] = nation_data['APPRENTICE_STATUS_CODE'].map(status_dict)

pd.DataFrame({
    'count': nation_data['APPRENTICE_STATUS_CODE'].value_counts(),
    'pct'  : nation_data['APPRENTICE_STATUS_CODE'].value_counts(normalize=True).mul(100).round(2)
})

,count,pct
APPRENTICE_STATUS_CODE,,
Completed,754443,51.81
Cancelled,683557,46.94
Active,18018,1.24
Suspended,204,0.01
Reinstated,61,0.00


In [10]:
# Rename PROBATIONARY_CANCELLERS to a more intuitive name
# and create a human-readable label column
nation_data['EARLY_TRAINING_EXIT_TYPE'] = nation_data['PROBATIONARY_CANCELLERS']
nation_data = nation_data.drop(columns=['RETENTION_APPR', 'PROBATIONARY_CANCELLERS'])

nation_data['EARLY_TRAINING_EXIT_TYPE'] = (
    nation_data['EARLY_TRAINING_EXIT_TYPE']
    .replace({'Y': 1, 'N': 0})
    .astype('float')
    .fillna(-1)
)

exit_labels = {
    -1: 'Unknown',
     0: 'No Early Exit',
     1: 'Exited During Early Training',
     2: 'Exited After Early Training'
}
nation_data['EARLY_TRAINING_EXIT_LABEL'] = nation_data['EARLY_TRAINING_EXIT_TYPE'].map(exit_labels)

pd.DataFrame({
    'count': nation_data['EARLY_TRAINING_EXIT_LABEL'].value_counts(),
    'pct'  : nation_data['EARLY_TRAINING_EXIT_LABEL'].value_counts(normalize=True).mul(100).round(2)
})

,count,pct
EARLY_TRAINING_EXIT_LABEL,,
Unknown,1069747,73.46
Exited After Early Training,232774,15.98
No Early Exit,152316,10.46
Exited During Early Training,1446,0.10


In [11]:
# Sanity check
apprentice_cols = [
    'APPRENTICE_STATUS_CODE',
    'ACTUAL_COMPLETER', 'ACTIVE_APPR', 'CANCELLED_APPR', 'SUSPENDED_APPR',
    'EARLY_TRAINING_EXIT_TYPE', 'EARLY_TRAINING_EXIT_LABEL'
]
nation_data[apprentice_cols].sample(3)

,APPRENTICE_STATUS_CODE,ACTUAL_COMPLETER,ACTIVE_APPR,CANCELLED_APPR,SUSPENDED_APPR,EARLY_TRAINING_EXIT_TYPE,EARLY_TRAINING_EXIT_LABEL
401720,Completed,1,0,0,0,-1.0,Unknown
555883,Cancelled,0,0,1,0,-1.0,Unknown
296425,Cancelled,0,0,1,0,0.0,No Early Exit


## 4. Demographic Columns

- `ETHNICITY` dropped: only captures Hispanic/non-Hispanic — `RACE_AND_ETHNICITY` is more complete
- `UNDERREPRESENTED` and `RACE` dropped: redundant given `RACE_AND_ETHNICITY`

In [12]:
nation_data = nation_data.drop(columns=['UNDERREPRESENTED', 'ETHNICITY', 'RACE'])

demographic_cols = [
    'EDUCATION', 'SEX', 'RACE_AND_ETHNICITY',
    'VETERAN_STATUS', 'INDIVIDUALS_WITH_DISABILITIES',
    'AGE_COHORT', 'YOUTH'
]
nation_data[demographic_cols].sample(3)

,EDUCATION,SEX,RACE_AND_ETHNICITY,VETERAN_STATUS,INDIVIDUALS_WITH_DISABILITIES,AGE_COHORT,YOUTH
507840,Bachelor's degree,Male,Hispanic or Latino,Non Veteran,No,Ages 24 and Under,1
800522,High School graduate (including equivalency),Male,White,Non Veteran,No,Ages 25-54,0
236299,High School graduate (including equivalency),Male,Black or African American,Non Veteran,Participant Did Not Self-Identify,Ages 25-54,0


In [13]:
# Distribution check
pd.DataFrame({
    'count': nation_data['AGE_COHORT'].value_counts(),
    'pct'  : nation_data['AGE_COHORT'].value_counts(normalize=True).mul(100).round(2)
})

,count,pct
AGE_COHORT,,
Ages 25-54,943646,64.80
Ages 24 and Under,474168,32.56
Ages 55+,37987,2.61
Participant Did Not Self-Identify,482,0.03


## 5. Occupation & Industry Columns

- `ONET_SOC_CD` is the standardized federal job code — use for merging and filtering
- `OCCUPATION` is free-text employer entry — too messy for analysis, dropped
- `RAPIDS_CD` has 18.7% missing and inconsistent formatting — dropped

### Data quality — missing ONET investigation

In [14]:
# Missing counts across occupation fields
missing_cols = [
    'ONET_SOC_CD', 'RAPIDS_CD', 'OCCUPATION_GROUP_NAME',
    'BROAD_GROUP_NAME', 'MINOR_GROUP_NAME', 'MAJOR_GROUP_NAME'
]
pd.DataFrame({
    'missing'    : nation_data[missing_cols].isna().sum(),
    'pct_missing': nation_data[missing_cols].isna().mean().mul(100).round(1)
})

,missing,pct_missing
ONET_SOC_CD,169963,11.7
RAPIDS_CD,272404,18.7
OCCUPATION_GROUP_NAME,178479,12.3
BROAD_GROUP_NAME,178479,12.3
MINOR_GROUP_NAME,178479,12.3
MAJOR_GROUP_NAME,178479,12.3


In [15]:
# When ONET is missing, all occupation fields are missing together
nation_data.loc[nation_data['ONET_SOC_CD'].isna(), missing_cols].head(3)

,ONET_SOC_CD,RAPIDS_CD,OCCUPATION_GROUP_NAME,BROAD_GROUP_NAME,MINOR_GROUP_NAME,MAJOR_GROUP_NAME
0,None,None,None,None,None,None
12,None,None,None,None,None,None
13,None,None,None,None,None,None


In [16]:
# 99.9% of missing ONET concentrated in 7 states with legacy reporting systems
top_missing_ONET_states = ['CA', 'WA', 'NY', 'MA', 'NM', 'MD', 'NC']
missing_top_states = nation_data.loc[
    nation_data['STATE'].isin(top_missing_ONET_states), 'ONET_SOC_CD'
].isna().sum()

total_missing = nation_data['ONET_SOC_CD'].isna().sum()
print(f"Missing in top 7 states:    {missing_top_states:,}")
print(f"Total missing nationally:   {total_missing:,}")
print(f"Share outside top 7 states: {((total_missing - missing_top_states) / total_missing * 100):.2f}%")

Missing in top 7 states:    169,837
Total missing nationally:   169,963
Share outside top 7 states: 0.07%


In [17]:
# % missing by state
nation_data.groupby('STATE')['ONET_SOC_CD'].apply(
    lambda x: round(x.isna().sum() / len(x) * 100, 1)
).sort_values(ascending=False).head(10)

STATE
NM    97.7
NY    93.8
WA    87.7
MA    80.8
MD    29.6
CA    26.2
NC     9.0
VT     1.5
MN     0.3
RI     0.3
Name: ONET_SOC_CD, dtype: float64

In [18]:
# Maps: missing rate within each state vs share of national missing
state_missing = nation_data.groupby('STATE', as_index=False)['ONET_SOC_CD'].agg(
    total   = lambda x: len(x),          # ← total rows including NaN
    missing = lambda x: x.isna().sum()
)
state_missing['pct_missing_in_state']    = (state_missing['missing'] / state_missing['total'] * 100).round(1)
state_missing['pct_of_national_missing'] = (state_missing['missing'] / state_missing['missing'].sum() * 100).round(1)

px.choropleth(
    state_missing,
    locations             = 'STATE',
    locationmode          = 'USA-states',
    color                 = 'pct_missing_in_state',
    scope                 = 'usa',
    color_continuous_scale = 'Reds',
    title                 = '% missing ONET_SOC_CD within each state',
    labels                = {'pct_missing_in_state': '% missing'}
).show()

px.choropleth(
    state_missing,
    locations             = 'STATE',
    locationmode          = 'USA-states',
    color                 = 'pct_of_national_missing',
    scope                 = 'usa',
    color_continuous_scale = 'Blues',
    title                 = 'State share of all missing ONET records nationally',
    labels                = {'pct_of_national_missing': '% of national missing'}
).show()

In [19]:
# Hawaii has 0% missing — wage forecast merge will be complete
hawaii_check = nation_data[nation_data['STATE'] == 'HI']
print(f"Hawaii records:      {len(hawaii_check):,}")
print(f"Hawaii ONET missing: {hawaii_check['ONET_SOC_CD'].isna().mean() * 100:.1f}%")

Hawaii records:      14,238
Hawaii ONET missing: 0.0%


**Data Quality Note — Missing ONET_SOC_CD**

169,963 records (11.7%) are missing occupation codes nationally.
99.9% originate from 7 states with inconsistent federal reporting:
CA, WA, NY, MA, NM, MD, NC. These states use systems that predate
RAPIDS and do not fully sync occupation codes.

This is an administrative data gap, not a program quality issue.
Hawaii has 0% missing — the wage forecast merge will be complete.

### Cleaning

In [20]:
# Drop messy free-text and unreliable code columns
nation_data = nation_data.drop(columns=['OCCUPATION', 'RAPIDS_CD'], errors='ignore')

# Standardize SOC format for merging ('11-1011.00' → '11-1011')
nation_data['SOC_MERGE_KEY'] = (
    nation_data['ONET_SOC_CD']
    .str[:7]
    .str.strip()
)

# Flag rows missing all occupation data
# Hawaii has 0% missing so this flag is only meaningful for national analysis
nation_data['HAS_SOC'] = nation_data['ONET_SOC_CD'].notna()

assert nation_data['SOC_MERGE_KEY'].str.len().dropna().eq(7).all(), \
    "SOC_MERGE_KEY should be exactly 7 characters"

## 6. Geography Columns

HUD columns are redundant with DOL region columns — dropped.  
RAPIDS uses 6 DOL regions (numbered 1–6) plus Region 8 for federal/military programs.  
Note: region names refer to DOL office locations, not geographic identity.

In [21]:
# Drop HUD columns — redundant with STATE and REGION
geo_drop_cols = ['HUD_STATE', 'HUD_ZIPCODE', 'HUD_REGION', 'HUD_REGION_NAME']
nation_data = nation_data.drop(columns=geo_drop_cols, errors='ignore')

# Fix dtypes
nation_data['STATE']       = nation_data['STATE'].astype('string')
nation_data['REGION']      = nation_data['REGION'].astype('Int64')
nation_data['REGION_NAME'] = nation_data['REGION_NAME'].astype('string')

# Add readable region labels (official DOL office names)
region_labels = {
    1: 'Region 1 — Boston',
    2: 'Region 2 — Philadelphia',
    3: 'Region 3 — Atlanta',
    4: 'Region 4 — Dallas',
    5: 'Region 5 — Chicago',
    6: 'Region 6 — San Francisco',
    8: 'Region 8 — Federal/Military'
}
nation_data['REGION_LABEL'] = nation_data['REGION'].map(region_labels).astype('string')

# Confirm Hawaii is in Region 6 as expected
nation_data[nation_data['STATE'] == 'HI']['REGION_LABEL'].value_counts()

REGION_LABEL
Region 6 — San Francisco    14238
Name: count, dtype: Int64

### 7. Date and Time

In [24]:
# 7. Date Columns (Currently as String Data Types)
date_cols = ['START_DATE',                       # When the apprentice started the program
             'EXIT_DATE',                        # When they left the program (completion or cancellation)
             'EXPECT_COMPLETE_DT'                # Expected completion date 
]

# Cleaning date columns to become datetime variable types 
nation_data[date_cols] = nation_data[date_cols].apply(
    lambda col: pd.to_datetime(col, errors='coerce').dt.floor('D')
)
               
nation_data['duration_days'] = (
    nation_data['EXIT_DATE'] - nation_data['START_DATE']
).dt.days

# 2. Remove negative durations (data entry errors)
nation_data.loc[nation_data['duration_days'] < 0, 'duration_days'] = np.nan

# 3. Remove impossible durations (> 10 years)
nation_data.loc[nation_data['duration_days'] > 365*10, 'duration_days'] = np.nan

# 4. Remove zero-day durations (same-day cancellation)
nation_data.loc[nation_data['duration_days'] == 0, 'duration_days'] = np.nan

nation_data['duration_months'] = nation_data['duration_days'] / 30.44


# Cleaning date columns to become datetime variable types 
nation_data[date_cols] = nation_data[date_cols].apply(
    lambda col: pd.to_datetime(col, errors='coerce').dt.floor('D')
)

# EXPECT_COMPLETE_DATE_ORIGINAL is identical to EXPECT_COMPLETE_DT for 100% of records in this dataset. 
# Programs rarely update expected completion dates.
# Drop the original to avoid confusion and reduce schema noise.
nation_data = nation_data.drop(columns=['EXPECT_COMPLETE_DATE_ORIGINAL'])

In [25]:
print(len(nation_data.loc[nation_data['duration_days'] > 365*10, 'duration_days']))

0


In [26]:
print(nation_data['duration_days'].describe().round(2))
print((nation_data['duration_days'] <= 0).sum())
print(min(nation_data['duration_days']))

count    1425689.00
mean         941.16
std          668.34
min            1.00
25%          366.00
50%          780.00
75%         1456.00
max         3650.00
Name: duration_days, dtype: float64
0
1.0


In [27]:
# 7b. Time Columns (Fiscal Year and Program Hours)

# FISCAL_YEAR is the fiscal year of registration; Rebrand type as category
nation_data['FISCAL_YEAR'] = nation_data['FISCAL_YEAR'].astype('Int64')
assert nation_data['FISCAL_YEAR'].dtype == 'Int64'

# PROGRAM_HOURS is the official required hours for the apprenticeship program
# Rebranded from less explict measurement of time of TERMLENGTH
nation_data = nation_data.rename(columns={'TERMLENGTH': 'PROGRAM_HOURS'})
nation_data['PROGRAM_HOURS'] = nation_data['PROGRAM_HOURS'].replace(0, np.nan)

# YEARS_TO_COMPLETE converts from PROGRAM_YEARS and it reflects 1 year of apprenticeship = 2,000 hours
nation_data['YEARS_TO_COMPLETE'] = nation_data['PROGRAM_HOURS'] / 2000

assert nation_data['PROGRAM_HOURS'].dtype == 'float64'
assert nation_data['YEARS_TO_COMPLETE'].dtype == 'float64'

### **8. Sponsor Type**

* Indicate what type of organization sponsors the apprenticeship.
* These are 1/0 flags. A program can have multiple sponsor types.

In [6]:
sponsor_cols = ['SPON_EMPLOYER', 'SPON_UNIONLABOR', 'SPON_BSNSSASSCTN',
                'SPON_INTERMEDIARY', 'SPON_CMMNTYCLLGUNIV', 'SPON_CMNTYBSDORG',
                'SPON_WRKFRCDVLPMNTBRD', 'SPON_FOUNDATION', 'SPON_FEDERALAGENCY',
                'SPON_STATEAGENCY', 'SPON_CITYCOUNTYAGENCY', 'SPON_OTHER', 'SPON_ALL_SPONSOR_TYPE']

nation_data[sponsor_cols].head()

,SPON_EMPLOYER,SPON_UNIONLABOR,SPON_BSNSSASSCTN,SPON_INTERMEDIARY,SPON_CMMNTYCLLGUNIV,SPON_CMNTYBSDORG,SPON_WRKFRCDVLPMNTBRD,SPON_FOUNDATION,SPON_FEDERALAGENCY,SPON_STATEAGENCY,SPON_CITYCOUNTYAGENCY,SPON_OTHER,SPON_ALL_SPONSOR_TYPE
0,0,0,0,0,0,0,0,0,0,0,0,0,None
1,1,0,0,0,0,0,0,0,0,0,0,0,Employer
2,0,0,1,0,0,0,0,0,0,0,0,0,Business Association
3,0,0,0,0,0,0,0,0,0,0,0,0,None
4,0,0,0,0,0,0,0,0,0,1,0,0,State Agency


### **9. Special populations**
* `INMATE_IND` = incarcerateed apprentice
* `UNION_Y_N` = union-affiliated

In [9]:
nation_data['UNION_Y_N'] = nation_data['UNION_Y_N'].astype('Int64')

In [ ]:
nation_data[nation_data['PROGRAM_NAME'] == "Pearl Harbor Naval Shipyard & IMF"]['UNION_Y_N'].value_counts()

343        1
508        1
127667     1
127671     1
127672     1
          ..
1494129    1
1494130    1
1494131    1
1494132    1
1494133    1
Name: UNION_Y_N, Length: 1931, dtype: Int64

## **10. Export**

Save full national dataset and Hawaii subset separately.  
Raw files are never modified; only processed versions are saved here.

In [30]:
# Save national dataset for future expansion
national_data = nation_data.copy()

# Save Hawaii subset
hawaii_data = nation_data[nation_data['STATE'] == 'HI'].copy()

print(f"National records: {len(national_data):,}")
print(f"Hawaii records:   {len(hawaii_data):,}")
print(f"Columns:          {national_data.shape[1]}")

National records: 1,456,283
Hawaii records:   14,238
Columns:          62


In [31]:
national_data.to_parquet('../data/processed/national_apprenticeships_clean.parquet', index=False)
hawaii_data.to_parquet('../data/processed/hawaii_apprenticeships_clean.parquet', index=False)

print("Exported:")
print("  ../data/processed/national_apprenticeships_clean.parquet")
print("  ../data/processed/hawaii_apprenticeships_clean.parquet")

Exported:
  ../data/processed/national_apprenticeships_clean.parquet
  ../data/processed/hawaii_apprenticeships_clean.parquet
